# Validation: Spark and Catalog

## Introduction
This notebook will be used to check if connection among Spark and Catalog 
(Project Nessie - MinIo) is working. 

In [ ]:
print('Hello world, python is running!')

## Spark

In [ ]:
import os 
s3_access_key = os.getenv('MINIO_ROOT_USER')
s3_secret_ket = os.getenv('MINIO_ROOT_PASSWORD')

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Delta Test")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", s3_access_key)
    .config("spark.hadoop.fs.s3a.secret.key", s3_secret_ket)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.delta.logStore.class", "org.apache.spark.sql.delta.storage.S3SingleDriverLogStore")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

df = spark.createDataFrame([("Alice", 1), ("Bob", 2)], ["name", "id"])
df.write.format("delta").mode("overwrite").save("s3a://bronze/test-delta-table_again")

In [ ]:
spark.sql("SELECT * FROM delta.`s3a://bronze/test-delta-table_again`").show()

In [ ]:
spark.sql("DROP TABLE IF EXISTS delta.`s3a://bronze/test-delta-table_again`")